In [2]:
import asyncio
from runtime_support import (
    setup_client_from_env,
    build_fleet_object,
    api_navigate_ship,
    unwrap_data)
from core_helpers import (
    init_world_state
)

from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi

with setup_client_from_env() as client:
        fleet_api = FleetApi(client)
        agents_api = AgentsApi(client)
        systems_api = SystemsApi(client)

ships_activity_obj = await build_fleet_object(fleet_api)
world_state = await init_world_state(fleet_api, agents_api, systems_api)

[BOOT] Adapted 2 ships into fleet_object


In [2]:
print(world_state)

In [3]:
#----------------- db init for fleet ----------------------
from db.auto_repo_sqlite import TableSpec, upsert_many
import sqlite3
from adapters.ships_activity_adapter import merge_activity_with_nav
from adapters.ships_specs_adapter import adapt_ships_specs_from_ship
from core_helpers import adapt_ships_activity_from_ship
from typing import Any, Dict, Iterable, List, Optional, Tuple
from runtime_support import call_sdk
from dataclasses import dataclass, field
from domain.ships_activity import ShipsActivity
from domain.ships_specs import ShipsSpecs
from domain.ship_market import ShipMarketRow

# -------------------------- local state ---------------------------------------
@dataclass
class FleetState:
    """Local, easily-referenced state for your session."""
    # Activity & Specs keyed by ship symbol
    activities: Dict[str, ShipsActivity] = field(default_factory=dict)
    specs: Dict[str, ShipsSpecs] = field(default_factory=dict)

    # Shipyard listings cached by waypoint
    ship_market: Dict[str, List[ShipMarketRow]] = field(default_factory=dict)

    def ensure_activity(self, symbol: str) -> ShipsActivity:
        a = self.activities.get(symbol)
        if a is None:
            a = ShipsActivity(symbol=symbol)
            self.activities[symbol] = a
        return a

    def update_activity_from_nav(self, symbol: str, nav_dto: Any) -> ShipsActivity:
        current = self.ensure_activity(symbol)
        updated = merge_activity_with_nav(current, nav_dto)
        self.activities[symbol] = updated
        return updated

    def update_activity_from_refuel(self, symbol: str, resp_dto: Any) -> ShipsActivity:
        """Use refuel response to update fuel state in local activity."""
        current = self.ensure_activity(symbol)
        fuel_cur = getattr(getattr(resp_dto, "fuel", None), "current", None)
        fuel_cap = getattr(getattr(resp_dto, "fuel", None), "capacity", None)
        patched = current.model_copy(update={
            "fuel_current": fuel_cur if fuel_cur is not None else current.fuel_current,
            "fuel_capacity": fuel_cap if fuel_cap is not None else current.fuel_capacity,
        })
        self.activities[symbol] = patched
        return patched
# -------------------------- fleet state writer --------------------------------
async def load_initial_fleet_state(conn: sqlite3.Connection, fleet: FleetApi) -> FleetState:
    """Load ships once, build local state (activity + specs) and persist to DB."""
    resp = await call_sdk(fleet, "get_my_ships")
    ships: Iterable[Any] = unwrap_data(resp)
    ships = list(ships)
    if not ships:
        raise SystemExit("[FATAL] No ships returned; check token/agent.")

    activities = [adapt_ships_activity_from_ship(d) for d in ships]
    specs = [adapt_ships_specs_from_ship(d) for d in ships]

    # Persist to DB (idempotent upserts)
    upsert_many(conn, TableSpec(table="ships_activity", pk="symbol"), activities)
    upsert_many(conn, TableSpec(table="ships_specs", pk="symbol"), specs)

    # Build local state dicts
    state = FleetState(
        activities={a.symbol: a for a in activities if a.symbol},
        specs={s.symbol: s for s in specs if s.symbol},
    )
    return state

#-------- connect and write -------------
conn = sqlite3.connect("spacetraders.db")
# fleet activity and specs write-to-db
state = await load_initial_fleet_state(conn, fleet_api)


In [ ]:
# --------- define ship roles -----------

print(state.specs)
# Extract symbol for a given role
def get_symbol_by_role(ships_dict, role):
    for ship in ships_dict.values():
        if ship.role == role:
            return ship.symbol
    return None

command_ship = get_symbol_by_role(state.specs, "COMMAND")
sattelite = get_symbol_by_role(state.specs, "SATTELITE")
print(command_ship)

{'KIJINIBIBI-1': ShipsSpecs(symbol='KIJINIBIBI-1', role='COMMAND', frame_name='Frigate', frame_module_slots=8, frame_mounting_points=5, engine_name='Ion Drive II', speed=36, mounts=['MOUNT_SENSOR_ARRAY_II', 'MOUNT_GAS_SIPHON_II', 'MOUNT_MINING_LASER_II', 'MOUNT_SURVEYOR_II'], modules=['MODULE_CARGO_HOLD_II', 'MODULE_CREW_QUARTERS_I', 'MODULE_CREW_QUARTERS_I', 'MODULE_MINERAL_PROCESSOR_I', 'MODULE_GAS_PROCESSOR_I'], capacity=40), 'KIJINIBIBI-2': ShipsSpecs(symbol='KIJINIBIBI-2', role='SATELLITE', frame_name='Probe', frame_module_slots=0, frame_mounting_points=0, engine_name='Impulse Drive I', speed=9, mounts=[], modules=[], capacity=0)}
KIJINIBIBI-1


In [9]:
#------ db init for waypoints ------
from domain.waypoint_ref import WaypointRef
from domain.waypoint_trait import WaypointTraitRow
from adapters.waypoint_adapter import adapt_waypoints
from adapters.waypoint_trait_adapter import adapt_traits_from_waypoint_dtos

from runtime_support import api_get_system_waypoints

from openapi_client.api.systems_api import SystemsApi
from openapi_client.api.agents_api import AgentsApi


async def load_initial_waypoint_state(
    conn: sqlite3.Connection,
    agents: AgentsApi,
    systems: SystemsApi,
) -> Tuple[List[WaypointRef], List[WaypointTraitRow]]:
    """
    Resolve the agent HQ's system, pull all waypoints in that system,
    adapt to domain rows, and persist to DB (idempotent upserts).
    Returns (waypoint_refs, waypoint_traits).
    """

    # 1) Figure out which system to index from agent HQ
    agent_resp = await call_sdk(agents, "get_my_agent")
    agent = unwrap_data(agent_resp)
    hq_wp = getattr(agent, "headquarters", None)
    if not hq_wp or "-" not in str(hq_wp):
        raise RuntimeError("Agent headquarters missing or malformed, cannot derive system.")
    sys_symbol = "-".join(str(hq_wp).split("-")[:2])  # e.g. "X1-HA25"

    # 2) Pull the system waypoints (DTO list)
    #    NOTE: If your client uses a different method name (e.g. list_system_waypoints),
    #          swap it here. The signature usually takes system_symbol, plus page/limit.
    
    wps_resp = await api_get_system_waypoints(systems, system_symbol=sys_symbol)
    wps_dtos: Iterable[Any] = unwrap_data(wps_resp)

    # 3) Adapt → domain types
    waypoint_refs: List[WaypointRef] = adapt_waypoints(wps_dtos)
    waypoint_traits: List[WaypointTraitRow] = adapt_traits_from_waypoint_dtos(wps_dtos)

    # 4) Persist to DB (idempotent upserts)
    upsert_many(conn, TableSpec(table="waypoint_refs", pk="symbol"), waypoint_refs)

    # If your TableSpec supports composite PKs, use the tuple form below.
    # If not, create a UNIQUE index on (waypoint_symbol, trait_symbol) in schema, and keep pk="id" if you have one.
    upsert_many(conn, TableSpec(table="waypoint_traits", pk=("waypoint_symbol", "trait_symbol")), waypoint_traits)

    conn.commit()
    return waypoint_refs, waypoint_traits


In [10]:
conn = sqlite3.connect("spacetraders.db")

# waypoints + traits for the agent's home system
wp_refs, wp_traits = await load_initial_waypoint_state(conn, agents_api, systems_api)
print(f"Persisted {len(wp_refs)} waypoint refs and {len(wp_traits)} traits.")

Persisted 91 waypoint refs and 267 traits.
